## Project path & imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

# Point Python at your repo root (adjust if your notebook sits elsewhere)
project_root = Path.cwd()
if not (project_root / "fiberphotometry").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Pull the pieces we need from the new package
from fiberphotometry.turnlab.context import set_context
from fiberphotometry.turnlab.linking import add_session_idx
from fiberphotometry.turnlab.detect import make_turn_syllable_series
from fiberphotometry.turnlab.cache_index import build_or_load_turn_index, slice_by_percentile, onset_mask_from_slice
from fiberphotometry.turnlab.peth_plot import compute_turn_peths_from_merged
from fiberphotometry.turnlab.experiments import exp, run_turn_experiment, run_turn_experiments
from fiberphotometry.turnlab.config import SIGNAL_COLS, SAMPLING_RATE

## Load state (sessions, merged_all)

In [3]:
import pickle

LOAD_PKL = "big_run_frac_1_ld10_jn24_with_merged.pkl"  # <— set your path

with open(LOAD_PKL, "rb") as f:
    state = pickle.load(f)

sessions        = state["sessions"]
merged_all      = state["merged_all"]
PROJECT_DIR_FULL = state.get("project_dir_full")

print(f"✅ Loaded: sessions={len(sessions)}, merged_all={merged_all.shape}")

✅ Loaded: sessions=16, merged_all=(3544889, 26)


## Provide context to the modules & link sessions

In [4]:
# Let plotting/meta helpers access sessions & project folder
set_context(sessions, PROJECT_DIR_FULL)

# Attach session_idx (bijective match to kpms_h5_path stem)
merged_all = add_session_idx(merged_all, sessions)
merged_all.head(2)

,session_base,name,group,syllable,onset,frame_index,in_video_index,had_signal_match,centroid_x,centroid_y,...,signal_0_470_dff_z,signal_1_470_dff,signal_1_470_dff_z,signal_0_415,signal_1_415,curr_video_frame,current_h5_row,syllable_fp,moseq_frame_index,session_idx
0,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,WT,20,True,0,True,True,168.285395,203.203868,...,0.492600,0.19116,0.938041,NaN,NaN,69521,69521,20.0,0,1
1,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,WT,20,False,1,True,True,168.234610,203.261035,...,0.288502,NaN,0.928006,0.013308,0.027389,69522,69522,20.0,1,1


## (Optional) prune + downcast to keep RAM under control

In [5]:
import pandas as pd

_need_detect = {'session_idx','frame_index','group','angular_velocity','velocity_px_s'}
_need_peth   = set(SIGNAL_COLS)
_keep        = list((_need_detect | _need_peth) & set(merged_all.columns))

before_mb = merged_all.memory_usage(deep=True).sum() / (1024**2)
merged_all = merged_all.loc[:, _keep].copy()

merged_all['session_idx']     = pd.to_numeric(merged_all['session_idx'], downcast='integer')
merged_all['frame_index']     = pd.to_numeric(merged_all['frame_index'], downcast='integer')
merged_all['group']           = merged_all['group'].astype('category')
merged_all['angular_velocity']= pd.to_numeric(merged_all['angular_velocity'], errors='coerce').astype('float32')
merged_all['velocity_px_s']   = pd.to_numeric(merged_all['velocity_px_s'], errors='coerce').astype('float32')
for col in SIGNAL_COLS:
    merged_all[col] = pd.to_numeric(merged_all[col], errors='coerce').astype('float32')

after_mb = merged_all.memory_usage(deep=True).sum() / (1024**2)
print(f"🧹 Memory: {before_mb:.1f} → {after_mb:.1f} MB; kept {list(merged_all.columns)}")

🧹 Memory: 1970.6 → 74.4 MB; kept ['velocity_px_s', 'group', 'session_idx', 'frame_index', 'signal_0_470_dff_z', 'signal_1_470_dff_z', 'angular_velocity']


## Quick: detect turn onsets & attach columns

In [7]:
from fiberphotometry.turnlab.initiation import make_initiation_series

In [10]:
# ---- define presets ----
PRESETS = {
    "A": dict(min_rest_ms=100, min_move_ms=60,  sd_mult=1.5, speed_floor=30.0),
    "B": dict(min_rest_ms=200, min_move_ms=80,  sd_mult=1.5, speed_floor=25.0),
}

# (mode_tag, lateralization, extra_plot_kwargs, dir_suffix)
MODES = [
    ("rel", "relative", {"collapse_relative": "all"}, "rel"),
    ("abs", "absolute", {},                                "abs"),
]

# make sure we don't bloat the original object
merged_all = merged_all.copy()

results = {}
for tag, params in PRESETS.items():
    # 1) detect onsets/labels for this preset
    lbl, on = make_initiation_series(
        merged_all,
        session_idx_col='session_idx',
        frame_col='frame_index',
        speed_col='velocity_px_s',
        omega_col='angular_velocity',
        fps=SAMPLING_RATE,
        **params
    )
    label_col = f"turn_syllable_{tag}"
    onset_col = f"turn_onset_{tag}"
    merged_all[label_col] = lbl
    merged_all[onset_col] = on

    # 2) run both lateralization modes
    for mode_tag, lat, extra_pk, dir_sfx in MODES:
        out_dir = (PROJECT_DIR_FULL or '.') + f"/peth_initiations_{dir_sfx}_preset{tag}"
        res = compute_turn_peths_from_merged(
            merged_all,
            desired_labels={'left_init','right_init'},
            label_col=label_col,
            onset_col=onset_col,
            save=True,
            out_dir=out_dir,
            combine_signals=True,
            lateralization=lat,
            link_y_by_region=True,
            **extra_pk  # adds collapse_relative='all' for relative, nothing for absolute
        )
        key = f"preset{tag}_{mode_tag}"
        results[key] = res
        print(f"{key}: saved to {res['out_dir']}")

# (optional) quick counts summary
for key, res in results.items():
    print(key, "→ onset_counts:", res.get('totals_plotted', 'see figures'))


💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_rel_presetA/rel_collapsed_all.png
presetA_rel: saved to ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_rel_presetA
💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetA/left_init.png
💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetA/right_init.png
presetA_abs: saved to ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetA
💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_rel_presetB/rel_collapsed_all.png
presetB_rel: saved to ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_rel_presetB
💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetB/left_init.png


/home/fsp585/miniforge3/envs/keypoint_moseq/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/fsp585/miniforge3/envs/keypoint_moseq/lib/python3.9/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/home/fsp585/Desktop/FiberphotometryCode/fiberphotometry/turnlab/peth_plot.py:251: RuntimeWarning: All-NaN slice encountered
  ymin = np.nanmin(lo); ymax = np.nanmax(hi)


💾 Saved: ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetB/right_init.png
presetB_abs: saved to ./kpms_all_sessions_frac_1_ld10_jn24/peth_initiations_abs_presetB
presetA_rel → onset_counts: {'__REL_ALL__': 3426}
presetA_abs → onset_counts: {'left_init': 1756, 'right_init': 1670}
presetB_rel → onset_counts: {'__REL_ALL__': 1648}
presetB_abs → onset_counts: {'left_init': 830, 'right_init': 818}


In [ ]:
# turn_syll, turn_on = make_turn_syllable_series(
#     merged_all,
#     session_idx_col='session_idx',
#     frame_col='frame_index',
#     omega_col='angular_velocity',
#     speed_col='velocity_px_s',
#     method='omega',
#     granularity_percentiles=[90],
#     fps=SAMPLING_RATE,
#     turn_start_pct=90,
#     stop_ratio=0.6,
#     min_bout_ms=100,
#     merge_gap_ms=120,
#     smooth_win=5,
#     clip_hi_pct=99.5,
#     speed_floor=2.0
# )

# merged_all = merged_all.copy()
# merged_all['turn_syllable'] = turn_syll
# merged_all['turn_onset']    = turn_on

# merged_all[['turn_syllable','turn_onset']].head(3)

## Plot PETHs (relative contra/ipsi, collapsed-by-bin)

In [ ]:
res = compute_turn_peths_from_merged(
    merged_all,
    desired_labels={'left_init','right_init'},
    label_col='turn_syllable',
    onset_col='turn_onset',
    save=True,
    out_dir=(PROJECT_DIR_FULL or '.') + '/peth_initiations_rel',
    combine_signals=True,
    lateralization='relative',
    collapse_relative='all',
    link_y_by_region=True
)


In [ ]:
import os
out_dir = os.path.join(PROJECT_DIR_FULL or '.', 'peth_turns_demo_rel')

res = compute_turn_peths_from_merged(
    merged_all,
    desired_labels={'left_90','right_90'},
    label_col='turn_syllable',
    onset_col='turn_onset',
    save=True,
    out_dir=out_dir,
    combine_signals=True,
    lateralization='relative',     # contra/ipsi
    link_y_by_region=True,
    collapse_relative='by_bin'     # one PNG per bin → rel_collapsed_90.png
)

print("Saved to:", res['out_dir'])

## (Optional) WT-only subset for cleaner comparisons

In [ ]:
merged_wt = merged_all[merged_all['group'].astype(str) == 'WT'].copy()
if hasattr(merged_wt['group'], 'cat'):
    merged_wt['group'] = merged_wt['group'].cat.remove_unused_categories()
merged_wt.reset_index(drop=True, inplace=True)

print("WT frames:", len(merged_wt), "sessions:", merged_wt['session_idx'].nunique())

## Build/load the turn index cache (peak |ω|, |κ| + argsort ranks)

In [ ]:
cache = build_or_load_turn_index(
    merged_wt,                      # or merged_all
    dataset_tag='merged_wt',        # name your dataset
    detector_kwargs=dict(           # ONLY non-percentile params matter for the cache key
        session_idx_col='session_idx',
        frame_col='frame_index',
        omega_col='angular_velocity',
        speed_col='velocity_px_s',
        method='omega',
        granularity_percentiles=[90],  # ignored by the cache hash
        fps=SAMPLING_RATE,
        turn_start_pct=90,
        stop_ratio=0.6,
        min_bout_ms=100,
        merge_gap_ms=120,
        smooth_win=5,
        clip_hi_pct=99.5,
        speed_floor=2.0
    )
)
turn_ix = cache['turn_index']
turn_ix.head(3)

## Use the cache to slice bins instantly & plot

In [ ]:
# Pick a percentile window on kappa (80–85), build onset mask, label minimally, and plot
sl = slice_by_percentile(turn_ix, metric='kappa', p_lo=80, p_hi=85)
if len(sl):
    mask_col = onset_mask_from_slice(merged_wt, sl, onset_col='turn_onset_sel')

    # Minimal labels just to distinguish directions in plotting
    dir_map = dict(zip(turn_ix['onset_df_idx'], turn_ix['dir']))
    lab = merged_wt.get('turn_syllable_bin')
    if lab is None:
        import pandas as pd
        lab = pd.Series([None]*len(merged_wt), index=merged_wt.index, dtype='object')
    sel = sl['onset_df_idx'].to_numpy()
    lab.loc[sel] = [f"{dir_map[i]}_80_85" if dir_map[i] in ('left','right') else None for i in sel]
    merged_wt['turn_syllable_bin'] = lab

    out_dir = os.path.join(PROJECT_DIR_FULL or '.', 'peth_bins_cache_demo_kappa_80_85')
    res_bin = compute_turn_peths_from_merged(
        merged_wt,
        desired_labels={'left_80_85','right_80_85'},
        label_col='turn_syllable_bin',
        onset_col='turn_onset_sel',
        save=True,
        out_dir=out_dir,
        lateralization='relative',
        collapse_relative='by_bin',
        link_y_by_region=True,
        combine_signals=True,
        min_traces_per_syllable=10
    )
    print("Saved to:", res_bin['out_dir'])
else:
    print("No bouts in 80–85% bin (kappa) with current cache.")


## Run multi-experiment sweeps (if you want)

In [ ]:
# ===== Option A binning sweep: ω@75%, κ@90%, merge gap = 80 ms =====

# choose which dataframe to use
DF_IN   = merged_wt          # or merged_all
TAG     = 'merged_wt'        # keep cache naming consistent with DF_IN

# bins: equal thirds of the top 40% → [60,73), [73,87), [87,100)
BINS_THIRDS = [60, 73, 87]

# detector settings (Option A)
START_PCT_OMEGA = 75
START_PCT_KAPPA = 90
MERGE_GAP_MS    = 80

# keep bout rules identical to avoid confounds (you can lower κ if you want)
MIN_BOUT_MS   = 120
STOP_RATIO    = 0.6
SMOOTH_WIN    = 5
SPEED_FLOOR_OMEGA = 2.0   # (unused by ω, kept for API symmetry)
SPEED_FLOOR_KAPPA = 10.0  # stabilize κ at low speeds

def _name(method, bins):  # nice folder suffix
    return f"{method}_{bins[0]}" if len(bins)==1 else f"{method}_bins_{'_'.join(map(str,bins))}"

def mk_exp(name, method, bins, start_pct, speed_floor):
    return exp(
        name=name,
        method=method,
        bins=bins,
        start_pct=start_pct,
        min_bout_ms=MIN_BOUT_MS,
        merge_gap_ms=MERGE_GAP_MS,
        smooth_win=SMOOTH_WIN,
        speed_floor=speed_floor,
        stop_ratio=STOP_RATIO,
        desired_labels=None,
        dataset_tag=TAG
    )

experiments = [
    mk_exp(_name('omega', BINS_THIRDS), 'omega', BINS_THIRDS, START_PCT_OMEGA, SPEED_FLOOR_OMEGA),
    mk_exp(_name('kappa', BINS_THIRDS), 'kappa', BINS_THIRDS, START_PCT_KAPPA, SPEED_FLOOR_KAPPA),
]

common_plot = dict(
    out_dir=os.path.join(PROJECT_DIR_FULL or '.', 'peth_turns_optionA_bins_60_73_87'),
    min_traces_per_syllable=12,
    combine_signals=True,
    lateralization='relative',     # contra vs ipsi
    link_y_by_region=True,
    collapse_relative='by_bin',    # one PNG per bin → rel_collapsed_60/73/87.png
    verbose=True
)

sweep_optA = run_turn_experiments(
    DF_IN,
    experiments,
    common_plot=common_plot,
    use_cache=True,                 # uses/creates cache keyed by TAG + detector params
    dataset_tag=TAG,
    verbose=True
)

# quick sanity: show counts per bin and save dirs
for name, res in sweep_optA.items():
    print(f"\n{name}  →  {res.get('out_dir')}")
    print("bin_counts   :", res.get('bin_counts', {}))     # pre-window slice counts
    print("onset_counts :", res.get('onset_counts', {}))   # actually used onsets per label


In [ ]:
# ==== Single-percentile PETHs (no binning): {50, 75, 82, 90, 95, 97} × {omega, kappa} ====

# DF_IN   = merged_wt          # or merged_all
# TAG     = 'merged_wt'        # cache tag aligned with DF_IN

DF_IN = merged_all
TAG = 'merged_all'

PCTS = [0, 25, 50, 75, 82, 90, 95, 97]   # single-percentile conditions

# Option A detector settings
START_PCT_OMEGA = 75
START_PCT_KAPPA = 90
MERGE_GAP_MS    = 80

MIN_BOUT_MS   = 120
STOP_RATIO    = 0.6
SMOOTH_WIN    = 5
SPEED_FLOOR_OMEGA = 2.0
SPEED_FLOOR_KAPPA = 10.0

def _name_single(method, p):
    return f"{method}_{int(p)}"

def mk_exp_single(method, p, start_pct, speed_floor):
    return exp(
        name=_name_single(method, p),
        method=method,
        bins=[int(p)],             # single “bin” → left_p/right_p labels
        start_pct=start_pct,       # Option A thresholds
        min_bout_ms=MIN_BOUT_MS,
        merge_gap_ms=MERGE_GAP_MS,
        smooth_win=SMOOTH_WIN,
        speed_floor=speed_floor,
        stop_ratio=STOP_RATIO,
        desired_labels={f"left_{int(p)}", f"right_{int(p)}"},  # be explicit
        dataset_tag=TAG
    )

# Build 12 experiments: each percentile for omega and kappa
experiments_single = []
for p in PCTS:
    experiments_single.append(mk_exp_single('omega', p, START_PCT_OMEGA, SPEED_FLOOR_OMEGA))
    experiments_single.append(mk_exp_single('kappa', p, START_PCT_KAPPA, SPEED_FLOOR_KAPPA))

common_plot_single = dict(
    out_dir=os.path.join(PROJECT_DIR_FULL or '.', 'peth_turns_optionA_percentiles_all'),
    min_traces_per_syllable=12,
    combine_signals=True,
    lateralization='relative',    # contra vs ipsi
    link_y_by_region=True,
    collapse_relative='by_bin',   # one PNG per condition → rel_collapsed_<p>.png
    verbose=True
)

sweep_single = run_turn_experiments(
    DF_IN,
    experiments_single,
    common_plot=common_plot_single,
    use_cache=True,
    dataset_tag=TAG,
    verbose=True
)

# Summarize counts per condition
print("\n=== Summary: counts per condition ===")
for name, res in sweep_single.items():
    print(f"\n{name}  →  {res.get('out_dir')}")
    # pre-window slice counts (by our 'by_bin' convention this collapses to a single key per run)
    print("bin_counts   :", res.get('bin_counts', {}))
    # actually used onsets per label (left_p/right_p)
    print("onset_counts :", res.get('onset_counts', {}))
